<a href="https://colab.research.google.com/github/hyunsoooo1103/KHUDA_Vacation_Session/blob/main/3%EC%B0%A8%EC%8B%9C_Olist_Project_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brazilian E-Commerce by Olist 🛒 - 데이터 분석 & 예측 모델링 프로젝트

이 프로젝트는 브라질 전자상거래 플랫폼 Olist의 데이터에 대해 탐색적 데이터 분석(EDA)을 수행하고, 비즈니스 성과에 영향을 주는 주요 요인을 파악하는 것을 목표로 합니다. 통계적 인사이트와 비즈니스 관점의 결론을 함께 도출합니다.


## 🎯 비즈니스 문제

- *Olist는 여러 판매자와 브라질 전역의 고객을 연결하는 온라인 마켓플레이스입니다.*
- *이 회사는 배송 지연이라는 문제에 직면해 있습니다.*
- *배송이 늦어지면 고객 만족도와 리뷰 점수가 떨어지며, 결국 이탈률이 높아집니다. 심지어 전반적인 매출과 브랜드 평판에 타격을 줄 수도 있습니다.*
- ***Olist는 어떻게 이 문제를 해결하고 고객의 신뢰를 회복할 수 있을까요?***


# 1️⃣ Importing Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2️⃣ Loading Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/olist_data'

df_customer = pd.read_csv(f'{BASE_PATH}/olist_customers_dataset.csv')
df_location = pd.read_csv(f'{BASE_PATH}/olist_geolocation_dataset.csv')
df_order_items = pd.read_csv(f'{BASE_PATH}/olist_order_items_dataset.csv')
df_order_payment = pd.read_csv(f'{BASE_PATH}/olist_order_payments_dataset.csv')
df_order_review = pd.read_csv(f'{BASE_PATH}/olist_order_reviews_dataset.csv')
df_orders = pd.read_csv(f'{BASE_PATH}/olist_orders_dataset.csv')
df_products = pd.read_csv(f'{BASE_PATH}/olist_products_dataset.csv')
df_translation = pd.read_csv(f'{BASE_PATH}/product_category_name_translation.csv')
df_sellers = pd.read_csv(f'{BASE_PATH}/olist_sellers_dataset.csv')

# 3️⃣ Data Clean



### 3.1 Customer Dataset

In [ ]:
df_customer.info()

In [ ]:
df_customer.isna().sum()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=df_customer['customer_state'].value_counts().head(10).index,
            y=df_customer['customer_state'].value_counts().head(10).values, palette='rocket')
plt.title('Top 10 States with Most Customers')
plt.xlabel('State')
plt.ylabel('Number of Customers')
plt.show()

### 3.2 Products Dataset


In [ ]:
df_products.info()

- `df translation`과 left join으로 포르투갈어 카테고리명을 영어로 변환합니다.
- 번역 없는 항목은 Unknown으로 대체합니다.

In [ ]:
df_products_eng = df_products.merge(df_translation, on='product_category_name', how='left')
df_products_eng['product_category_name_english'] = df_products_eng['product_category_name_english'].fillna('Unknown')

In [ ]:
df_products_eng.isna().sum()

In [ ]:
df_products_eng = df_products_eng.dropna(subset=['product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty'])

In [ ]:
df_products_eng.isna().sum()

In [ ]:
plt.figure(figsize=(12, 6))
top_categories = df_products_eng['product_category_name_english'].value_counts().head(10)
sns.barplot(x=top_categories.values, y=top_categories.index, palette='viridis')
plt.title('Top 10 Product Categories (by Number of Unique Items)')
plt.xlabel('Count of Products')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_products['product_weight_g'] / 1000, bins=30, kde=False, color='skyblue')
plt.title('Product Weight Distribution (kg)')
plt.xlabel('Weight (kg)')
plt.xlim(0, 30)
plt.show()

### 3.3 Sellers Dataset

In [ ]:
df_sellers.isna().sum()

In [ ]:
df_sellers.duplicated().sum()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=df_sellers['seller_state'].value_counts().head(10).index,
            y=df_sellers['seller_state'].value_counts().head(10).values, palette='rocket')
plt.title('Top 10 States with Most Sellers')
plt.xlabel('State')
plt.ylabel('Number of Sellers')
plt.show()

### 3.4 Order Itmes Dataset

In [ ]:
df_order_items.info()

In [ ]:
df_order_items['shipping_limit_date'] = pd.to_datetime(df_order_items['shipping_limit_date'])

In [ ]:
df_order_items['freight_ratio'] = df_order_items['freight_value'] / df_order_items['price']

In [ ]:
avg_ratio = df_order_items['freight_ratio'].mean()

print("평균 배송비 비율 (%):", avg_ratio * 100)

In [ ]:
items_per_order = (
    df_order_items
    .groupby('order_id')['order_item_id']
    .count()
    .reset_index(name='item_count')
)

item_distribution = (
    items_per_order['item_count']
    .value_counts()
    .sort_index()
    .reset_index()
)

item_distribution.columns = ['item_count', 'order_count']

plt.figure(figsize=(10, 6))
sns.histplot(items_per_order['item_count'], bins=range(1, items_per_order['item_count'].max() + 2), kde=False)
plt.title('Distribution of Items per Order')
plt.xlabel('Number of Items in Order')
plt.ylabel('Number of Orders')
plt.xticks(range(1, items_per_order['item_count'].max() + 1))
plt.grid(axis='y', alpha=0.75)
plt.show()

item_distribution

### 3.5 Order Payment Dataset

In [ ]:
df_order_payment.info()

In [ ]:
df_order_payment.isna().sum()

In [ ]:
df_order_payment['payment_type'].value_counts(normalize=True) * 100

### 3.6 Order Review Dataset

In [ ]:
df_order_review.info()

In [ ]:
df_order_review['review_creation_date']   = pd.to_datetime(df_order_review['review_creation_date'])
df_order_review['review_answer_timestamp'] = pd.to_datetime(df_order_review['review_answer_timestamp'])

In [ ]:
# Write your code here
df_order_review['response_time'] = (
    df_order_review['review_answer_timestamp'] - df_order_review['review_creation_date']
)

In [ ]:
avg_response_time = df_order_review['response_time'].mean()

print("평균 응답 시간:", avg_response_time)

### 3.7 Orders Dataset

In [ ]:
df_orders.info()

In [ ]:
df_orders.isna().sum()

In [ ]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

In [ ]:
for col in date_cols:
    df_orders = df_orders.dropna(subset=col)

### 3.8 Geolocation Dataset

In [ ]:
df_location.info()

# 4️⃣ Features Engineering

시간 정보 추출하기

In [ ]:
df_orders['purchase_year'] = df_orders['order_purchase_timestamp'].dt.year
df_orders['purchase_month'] = df_orders['order_purchase_timestamp'].dt.strftime('%Y-%m')
df_orders['purchase_day_of_week'] = df_orders['order_purchase_timestamp'].dt.day_name()

부피 피처 생성하기

In [ ]:
df_products_eng['product_volume_cm3'] = (
    df_products_eng['product_length_cm'] *
    df_products_eng['product_height_cm'] *
    df_products_eng['product_width_cm']
)

In [ ]:

# 1. 실제 배송 소요 기간 (도착일 - 구매일)
df_orders['delivery_days'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.total_seconds() / 86400

# 2. 지연 여부
df_orders['is_late'] = (df_orders['order_delivered_customer_date'] - df_orders['order_estimated_delivery_date']).dt.total_seconds().apply(lambda x: 1 if x > 3600 else 0)

# 5️⃣ Merging All Datasets

In [ ]:
# 0. 같은 주문 내 동일 상품(product_id) 중복은 첫 번째 행만 남기기
#    order_item_id 기준으로 정렬 후 첫 행 유지
order_items_dedup = (
    df_order_items
    .sort_values(['order_id', 'product_id', 'order_item_id'])
    .drop_duplicates(subset=['order_id', 'product_id'], keep='first')
)

# 1. Starting with the Orders table
df_merged = pd.merge(df_orders, order_items_dedup, on='order_id', how='left')

# 2. Merging Products
df_merged = pd.merge(df_merged, df_products_eng, on='product_id', how='left')

# 3. Merging Sellers
df_merged = pd.merge(df_merged, df_sellers, on='seller_id', how='left')

# 4. Merging Customers
df_merged = pd.merge(df_merged, df_customer, on='customer_id', how='left')

In [ ]:
review_agg = (df_order_review.groupby('order_id', as_index=False).agg(review_score=('review_score', 'first')))
df_merged = pd.merge(df_merged, review_agg, on='order_id', how='left')


- 같은 주문에 리뷰가 여러 개 있을 수 있으므로 review_score의 첫번째 값을 대표 값으로 만듭니다.

In [ ]:
payment_agg = df_order_payment.groupby('order_id')['payment_value'].sum().reset_index()
payment_agg.rename(columns={'payment_value': 'total_order_value'}, inplace=True)
df_merged = pd.merge(df_merged, payment_agg, on='order_id', how='left')


*   한 주문에 여러 결제 방법이 섞일 수 있으므로, order_id 기준으로 payment_value를 합산합니다.



In [ ]:
df_merged.info()

In [ ]:
pd.set_option('display.max_columns', None)
print("pandas 'display.max_columns' option has been set to None (display all columns).")

In [ ]:
display(df_merged.head(10))

In [ ]:
df_merged.isna().sum()

In [ ]:
# Write your code here
temp_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
    'product_volume_cm3', 'total_order_value', 'review_score']
df_merged = df_merged.dropna(subset=temp_cols)

df_merged['product_category_name'] = df_merged['product_category_name'].fillna('Unknown')
df_merged['product_category_name_english'] = df_merged['product_category_name_english'].fillna('Unknown')


In [ ]:
df_merged.isna().sum()

In [ ]:
df_merged.shape

# 6️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

## 고객 분석

### 신규 고객 VS 재구매 고객

In [ ]:
unique_customer_counts = df_customer['customer_unique_id'].value_counts()

customer_type_labels = unique_customer_counts.apply(lambda x: 'Returned' if x > 1 else 'First-time')

customer_type_distribution = customer_type_labels.value_counts()

plt.title("Returned Vs First_time Unique Customers",fontdict={'fontsize':15,'fontweight':600})
plt.pie(customer_type_distribution,normalize=True,startangle=90,
        labels=customer_type_distribution.index,autopct='%1.2f%%')
plt.show()

💡 Insight

- 약 96.88%의 고객이 첫 구매자이다.

- 재구매 고객은 3.12%에 불과해, 고객 충성도와 유지에 어려움이 있음을 시사한다.

- 구매 후 사용자 경험과 배송 신뢰도를 향상시키면 첫 구매 고객을 재구매 고객으로 전환하는 데 도움이 될 수 있다.

In [ ]:
# 1. df_merged에서 주문 단위 중복 제거
#    주문별로 고객, 구매월, 주문금액만 남김
df_temp = df_merged[['order_id', 'customer_unique_id', 'purchase_month', 'total_order_value']].drop_duplicates(subset=['order_id']).copy()

# 2. purchase_month를 datetime으로 변환
df_temp['order_month'] = pd.to_datetime(df_temp['purchase_month'])

# 3. 고객별 첫 구매월 계산
df_temp['first_purchase_month'] = df_temp.groupby('customer_unique_id')['order_month'].transform('min')

# 4. 신규 고객 여부 판별
df_temp['is_new'] = df_temp['order_month'].eq(df_temp['first_purchase_month'])

# 5. 월별 신규 고객 수 / 신규 고객 매출
new_monthly = df_temp[df_temp['is_new']].groupby('order_month').agg(
    new_customers_count=('customer_unique_id', 'nunique'),
    new_customers_value=('total_order_value', 'sum')
)

# 6. 월별 재구매 고객 수
return_monthly = df_temp[~df_temp['is_new']].groupby('order_month').agg(
    return_customers_count=('customer_unique_id', 'nunique')
)

# 7. 월별 시계열 테이블 생성
data_customer_timeseries = pd.concat([new_monthly, return_monthly], axis=1).fillna(0).reset_index().rename(columns={'order_month': 'year_month'}).sort_values('year_month')

# 8. 표시용 문자열 변환
data_customer_timeseries['year_month'] = data_customer_timeseries['year_month'].dt.strftime('%Y-%m')

# 9. 시각화
fig = go.Figure()
fig.add_trace(go.Bar(
    x=data_customer_timeseries["year_month"],
    y=data_customer_timeseries["new_customers_count"],
    name="Number of new customers",
    marker_color="mediumaquamarine"
))
fig.add_trace(go.Bar(
    x=data_customer_timeseries["year_month"],
    y=data_customer_timeseries["return_customers_count"],
    name="Number of returned customers",
    marker_color="powderblue"
))
fig.add_trace(go.Scatter(
    x=data_customer_timeseries["year_month"],
    y=data_customer_timeseries["new_customers_value"],
    name="Revenue from new customers",
    yaxis="y2",
    marker_color="indianred",
    mode="lines+markers"
))
fig.update_layout(
    title=dict(
        text="<b>New Customers And Return Customers By Month</b>",
        font=dict(size=12, family="Arial", color="black")
    ),
    plot_bgcolor="white",
    barmode="stack",
    yaxis=dict(side="left", showgrid=False, zeroline=True, showline=False, showticklabels=False),
    yaxis2=dict(side="right", overlaying="y", showgrid=False, zeroline=False, showline=False, showticklabels=False),
    xaxis=dict(showline=True, linecolor="rgb(204, 204, 204)", linewidth=2),
    legend=dict(orientation="h"),
    hovermode="x unified",
    annotations=[dict(
        text="Created By Thuan Dao.",
        xref="paper", yref="paper",
        x=1.05, y=-0.25,
        showarrow=False,
        font=dict(size=10, color="gray", family="Arial")
    )]
)
fig.show(renderer="colab")

💡 Insight
- 2017년 신규 고객 유치에 큰 성공을 거두었으며 주요 매출원이 되었다.
- 2017년부터 2018년 초까지 신규 고객 수가 꾸준히 증가하다가 2018년 중반 이후 감소하는 경향을 보인다.
- 신규 고객으로부터 발생하는 매출이 전체 매출에 상당한 부분을 차지하고 있어, 신규 고객 확보가 매출 성장에 중요한 요소임을 알 수 있다.

## 리뷰 분석

### 리뷰 점수 분포

In [ ]:
review_counts = df_order_review['review_score'].value_counts().sort_index()

plt.figure(figsize=(7,7))
plt.pie(review_counts, labels=review_counts.index, autopct='%1.1f%%', startangle=90, colors=plt.cm.tab20.colors)
plt.title('Customer Review Score Distribution')
plt.show()

💡 Insight
- 고객 10명 중 약 8명이 서비스에 만족(4점 이상)하고 있으며, 이는 전반적인 쇼핑 경험이 긍정적임을 의미한다.


- 부정적 리뷰의 비중을 줄이기 위해 지연 가능성이 높은 주문을 선제적으로 관리한다면 고객 충성도를 높이고 재구매율을 개선할 수 있다.

✔ 배송 지연 여부별 리뷰 평균


In [ ]:
review_by_late = df_merged.groupby("is_late")["review_score"].agg(["mean","count","std"])
print(review_by_late)

### 응답 시간 분포

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df_order_review['response_time'].dt.total_seconds() / 3600, bins=50, kde=True)
plt.title("Distribution of Response Time (Hours)")
plt.xlabel("Response Time (Hours)")
plt.ylabel("Frequency")
plt.show()

## 시간대별 분석

### 월별 주문/지연 분석

In [ ]:
# 월별 주문 수 / 지연 주문 수 집계
monthly_orders = df_orders.groupby('purchase_month').agg(
    total_orders=('order_id', 'nunique'),
    total_delays=('is_late', 'sum')
).reset_index()

fig = go.Figure()

# 월별 주문 수 (Bar Chart)
fig.add_trace(
    go.Bar(
        x=monthly_orders['purchase_month'],
        y=monthly_orders['total_orders'],
        name='Total Orders',
        marker_color='rgb(64,224,208)'
    )
)

# 월별 배송 지연 수 (Line Chart), 두 번째 Y축 사용
fig.add_trace(
    go.Scatter(
        x=monthly_orders['purchase_month'],
        y=monthly_orders['total_delays'],
        name='Total Delays',
        mode='lines+markers',
        yaxis='y2',
        marker_color='indianred'
    )
)

fig.update_layout(
    title="Monthly Orders and Delays Trend",
    xaxis_title="Month",
    yaxis=dict(
        title="Number of Orders",
        side="left",
        showgrid=False,
        zeroline=True
    ),
    yaxis2=dict(
        title="Number of Delays",
        side="right",
        overlaying="y",
        showgrid=False,
        zeroline=True
    ),
    plot_bgcolor="white",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=1.1),
    hovermode="x unified"
)

fig.show(renderer="colab")

💡 Insight
- 2017년 초 주문량이 급격히 증가했으며 2017년 11월 정점을 찍었다.
- 2018년 3월 배송 지연이 가장 많았다.

### 요일별 지연 분석

In [ ]:
day_order = [
    'Monday','Tuesday','Wednesday',
    'Thursday','Friday','Saturday','Sunday'
]

day_orders = (
    df_merged
    .assign(Day_name=df_merged['order_approved_at'].dt.day_name())
    .groupby('Day_name')
    .agg(
        total_order=('order_id','count'),
        total_delay=('is_late','sum')
    )
    .reindex(day_order)
    .reset_index()
)

day_orders

In [ ]:
fig = go.Figure()

# 요일별 총 주문 수 (Bar Chart)
fig.add_trace(
    go.Bar(
        x=day_orders['Day_name'],
        y=day_orders['total_order'],
        name='Total Orders',
        marker_color='rgb(64,224,208)'
    )
)

# 요일별 총 지연 수 (Line Chart), 두 번째 Y축 사용
fig.add_trace(
    go.Scatter(
        x=day_orders['Day_name'],
        y=day_orders['total_delay'],
        name='Total Delays',
        mode='lines+markers',
        yaxis='y2',
        marker_color='indianred'
    )
)

fig.update_layout(
    title="Daily Orders and Delays Trend",
    xaxis_title="Day of Week",
    yaxis=dict(
        title="Number of Orders",
        side="left",
        showgrid=False,
        zeroline=True
    ),
    yaxis2=dict(
        title="Number of Delays",
        side="right",
        overlaying="y",
        showgrid=False,
        zeroline=True
    ),
    plot_bgcolor="white",
    legend=dict(orientation="h", x=0.5, xanchor="center", y=1.1),
    hovermode="x unified"
)

fig.show(renderer="colab")

💡 Insight
- 화요일은 주문량과 지연 건수가 가장 많아, 주중 중반에 운영 과부하가 발생할 가능성이 있다.
- 주말에는 주문이 적어 지연도 비교적 완만하게 유지된다.
- 이는 주 초반에 물류 및 운영 최적화가 가장 필요하다는 점을 보여준다.


### 시간대별 분석

In [ ]:
# orders across the hours and Days
total_order=pd.pivot_table(data=df_merged,index=df_merged['order_approved_at'].dt.hour,
               columns=df_merged['order_approved_at'].dt.day_name(),values='order_id',aggfunc='count')
total_order

In [ ]:
plt.figure(figsize=(13,8))
plt.title("Total Orders over the Hour by weekdays",fontdict={'fontsize':20,'fontweight':560})
sns.heatmap(data=total_order,annot=True,fmt='.2f',cmap="viridis",linecolor='white',linewidths=0.5)
plt.xlabel("Weekday",fontsize=15)
plt.ylabel("Hours",fontsize=15)
plt.show()

💡 Insight
- 대부분의 주문은 오전 10시부터 오후 4시 사이에 집중되어 있다.
- 이른 아침과 심야 시간대에는 주문이 상대적으로 적다. 이는 일반적인 소비자의 쇼핑 패턴과 일치한다.
- 새벽 3~4시 주문이 급증하는데, 주문 승인 시간(order_approved_at)의 영향일 수 있다.


## 지역별 분석

In [ ]:
State_delay_sumry=df_merged.groupby(['seller_state']).agg(delayed_order=('is_late','sum'),
                                   total_order=('is_late','count')).reset_index()
State_delay_sumry['delay%']=round((State_delay_sumry['delayed_order']/State_delay_sumry['total_order'])*100,2)
State_delay_sumry

In [ ]:
reg_delay=pd.pivot_table(data=df_merged,index='customer_state',columns='seller_state',values='is_late',aggfunc='sum',fill_value=0)

In [ ]:
plt.figure(figsize=(20,8))
plt.title("Total Delayed Orders between Seller's State and Customer Region",
          fontdict={'fontsize':18,'fontweight':700})
sns.heatmap(data=reg_delay,annot=True,fmt='.0f',cmap="cividis",linecolor='white',
            linewidths=0.5,annot_kws={"weight": "bold"})
plt.xlabel("Seller's State",fontdict={'fontsize':14,'fontweight':600})
plt.xticks(fontweight=600)
plt.ylabel("Customer's Region",fontdict={'fontsize':14,'fontweight':600})
plt.yticks(fontweight=600)
plt.show()

💡 Insight
- 상파울루(SP) 판매자와 동남부 고객 간에 가장 높은 지연이 발생한다.
- 상파울루는 브라질 물류의 중심지로 거래량이 가장 많기 때문에 절대적인 지연 건수도 높게 나타난다.
- 몇몇 주를 제외하고는 Inter-state가 활발하지 않거나, 특정 루트에서만 배송 사고가 집중됨을 시사한다.

# 7️⃣ Machine Learning

## Predictive Modeling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from imblearn.pipeline import Pipeline
import joblib

features = ['price', 'freight_value']
cat_cols = ['product_category_name_english', 'customer_state', 'seller_state']

x = df_merged[features + cat_cols].copy()
y = df_merged['is_late']

x[cat_cols] = x[cat_cols].fillna('Unknown')
x[features] = x[features].fillna(0)

try:
    cat_transform = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    cat_transform = OneHotEncoder(handle_unknown='ignore', sparse=False)

preproc = ColumnTransformer([
    ('num', 'passthrough', features),
    ('cat', cat_transform, cat_cols)
])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=24,
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline([
    ('preprocessor', preproc),
    ('model', clf)
])

pipe.fit(x_train, y_train)

joblib.dump(pipe, "delay_prediction_model.pkl")
print("\n📂📂📂 Model is saved.")
print("-" * 120)

pred = pipe.predict(x_test)
pred_proba = pipe.predict_proba(x_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, pred))
print(f"ROC-AUC : {roc_auc_score(y_test, pred_proba):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, pred_proba):.4f}")
print(f"Train Score : {pipe.score(x_train, y_train):.4f}")
print(f"Test Score  : {pipe.score(x_test, y_test):.4f}")
print("_" * 120)